![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)

# <h1><center>Electrical Motor Visualizer on Colab <a href="https://colab.research.google.com/github/robomotic/mujoco/blob/motors/python/examples/electrical/demo_visualizer_colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

This notebook will:

1. fetch the remote `motors` branch,
2. install the released MuJoCo wheel from GitHub Releases,
3. run the electrical visualizer demo, and
4. expose a browser port so the viewer can be opened inside Google Colab.

> The notebook prefers released wheels and now supports Python `3.11` and `3.12` runtimes, selecting the matching wheel automatically. If the requested wheel is not yet published, it falls back to building MuJoCo from source in Colab.

In [ ]:
from pathlib import Path
import os
import platform
import sys

REPO_URL = 'https://github.com/robomotic/mujoco.git'
BRANCH = 'motors'
RELEASE_TAG = 'motors-wheel-v3.7.0-11'
REPO_DIR = Path('/content/mujoco')
VNC_PORT = 6080
DISPLAY_ID = ':1'

SUPPORTED_WHEELS = {
    (3, 11): 'mujoco-3.7.0-cp311-cp311-linux_x86_64.whl',
    (3, 12): 'mujoco-3.7.0-cp312-cp312-linux_x86_64.whl',
}

py_key = sys.version_info[:2]
if py_key not in SUPPORTED_WHEELS:
    raise RuntimeError(
        f'Unsupported Colab runtime Python {sys.version.split()[0]}. '
        'Please switch to Python 3.11 or 3.12 (Runtime → Change runtime type).'
    )
WHEEL_NAME = SUPPORTED_WHEELS[py_key]
WHEEL_URL = f'https://github.com/robomotic/mujoco/releases/download/{RELEASE_TAG}/{WHEEL_NAME}'

os.environ['REPO_URL'] = REPO_URL
os.environ['BRANCH'] = BRANCH
os.environ['REPO_DIR'] = str(REPO_DIR)
os.environ['WHEEL_URL'] = WHEEL_URL
os.environ['VNC_PORT'] = str(VNC_PORT)
os.environ['DISPLAY_ID'] = DISPLAY_ID

print(f'Python:   {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'Repo:     {REPO_URL}')
print(f'Branch:   {BRANCH}')
print(f'Release:  {RELEASE_TAG}')
print(f'Wheel:    {WHEEL_URL}')
print(f'Port:     {VNC_PORT}')


In [ ]:
%%bash
set -euxo pipefail
apt-get update || true
DEBIAN_FRONTEND=noninteractive apt-get install -y --fix-missing \
  git libgl1-mesa-glx libglfw3 libosmesa6 mesa-utils \
  xvfb fluxbox x11vnc websockify novnc \
  build-essential cmake ninja-build python3-dev python3-venv pkg-config \
  libgl1-mesa-dev libwayland-dev libxinerama-dev libxcursor-dev libxkbcommon-dev \
  libxrandr-dev libxi-dev
python3 -m pip install --upgrade pip
python3 -m pip install --no-cache-dir --upgrade \
  "$WHEEL_URL" \
  glfw PyOpenGL absl-py "etils[epath]" || {
  echo "Wheel install failed; falling back to source build."
  pip install --upgrade pip setuptools wheel build
  if [ ! -d "$REPO_DIR/.git" ]; then
    git clone --depth=1 --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
  fi
  cd "$REPO_DIR/python"
  python3 ./make_sdist.sh
  cd dist
  pip wheel --no-deps mujoco-*.tar.gz
  pip install --no-index mujoco-*.whl
}
python3 - <<'PY'
import mujoco
print('Installed MuJoCo version:', mujoco.__version__)
print('Bundled plugin dir:', mujoco.PLUGINS_DIR)
PY


In [ ]:
%%bash
set -euxo pipefail
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --depth=1 --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
fi
cd "$REPO_DIR"
git fetch origin "$BRANCH" --depth=1
git checkout -B "$BRANCH" FETCH_HEAD
git status --short --branch


In [ ]:
import pathlib, shutil, urllib.request, json

# Clear the mujoco spec cache so fresh JSONs are fetched from the community
# repos (robomotic/mujoco-motors, robomotic/mujoco-batteries).
# This is necessary because a previous Colab session may have cached an older
# version of the Faulhaber spec with stall_current=109 (the physical short-
# circuit current) instead of stall_current=10 (the drive electronics limit).
cache_dir = pathlib.Path.home() / '.mujoco' / 'cache'
if cache_dir.exists():
    shutil.rmtree(cache_dir)
cache_dir.mkdir(parents=True, exist_ok=True)
print(f'Motor/battery spec cache cleared: {cache_dir}')

# Verify the remote motor spec has the correct drive current limit.
MOTOR_URL = (
    'https://raw.githubusercontent.com/robomotic/mujoco-motors/master'
    '/motor_assets/faulhaber/faulhaber_2264w024bp4.json'
)
with urllib.request.urlopen(MOTOR_URL, timeout=15) as r:
    spec = json.loads(r.read())
stall_current = spec.get('stall_current')
print(f'Remote faulhaber_2264w024bp4.json  stall_current = {stall_current} A')
assert stall_current == 10.0, f'Expected 10.0 A, got {stall_current}'
print('Drive current limit OK (10 A).')


In [ ]:
import mujoco
import mujoco.viewer
import inspect
from mujoco.electrical import SingleEnvSimulation
from mujoco.electrical.electrical_motor import ElectricalMotor

print('Wheel import OK:', mujoco.__version__)
print('Bundled plugin dir:', mujoco.PLUGINS_DIR)
print()

# Diagnose which electrical_motor.py is actually loaded.
import mujoco.electrical.electrical_motor as _em_mod
print('electrical_motor loaded from:', _em_mod.__file__)
print()

# Check if the current clamp is present in the installed code.
src = inspect.getsource(ElectricalMotor.compute_control)
has_clamp = 'stall_current' in src
print('Current clamp (stall_current) in compute_control:', has_clamp)
if not has_clamp:
    print('  WARNING: clamp is MISSING — old wheel or wrong file on sys.path')
print()

# Quick one-step smoke test: run the Faulhaber motor at stall and check current.
from mujoco.electrical.database import MotorDatabase
spec = MotorDatabase().load('faulhaber_2264w024bp4')
print(f'stall_current from loaded spec: {spec.stall_current} A')
motor = ElectricalMotor(spec, kp=220.0, kd=18.0)
torque, current = motor.compute_control(
    pos=0.0, vel=0.0, pos_tgt=1.5708, vel_tgt=0.0,
    effort_tgt=0.0, dt=0.002, bus_voltage=24.0,
)
print(f'One-step stall current: {current:.3f} A  (expect ≤ {spec.stall_current} A)')
print(f'One-step torque:        {torque:.4f} N·m')


In [ ]:
import os
import subprocess
import time
from google.colab import output

os.environ['DISPLAY'] = DISPLAY_ID
os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'

_bg_processes = globals().get('_bg_processes', {})

def start_once(name, cmd, env=None):
    proc = _bg_processes.get(name)
    if proc is not None and proc.poll() is None:
        print(f'{name} already running (pid={proc.pid})')
        return proc
    proc = subprocess.Popen(cmd, env=env or os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    _bg_processes[name] = proc
    print(f'started {name} (pid={proc.pid})')
    return proc

start_once('xvfb', ['Xvfb', DISPLAY_ID, '-screen', '0', '1440x900x24', '-ac', '+extension', 'GLX', '+render'])
time.sleep(2)
start_once('fluxbox', ['fluxbox'], env={**os.environ, 'DISPLAY': DISPLAY_ID})
start_once('x11vnc', ['x11vnc', '-display', DISPLAY_ID, '-forever', '-shared', '-nopw', '-rfbport', '5901'])
start_once('novnc', ['websockify', '--web=/usr/share/novnc/', str(VNC_PORT), 'localhost:5901'])

print(f'Opening noVNC on port {VNC_PORT}...')
output.serve_kernel_port_as_iframe(VNC_PORT, path='/vnc.html?autoconnect=true&resize=scale', height=720)

In [ ]:
import os
import subprocess

env = os.environ.copy()
env['DISPLAY'] = DISPLAY_ID
env['LIBGL_ALWAYS_SOFTWARE'] = '1'

demo_cmd = [
    'python3',
    str(REPO_DIR / 'python/examples/electrical/demo_visualizer.py'),
    '--light',
    '--steps',
    '5000',
]

demo_proc = subprocess.Popen(demo_cmd, cwd=str(REPO_DIR), env=env)
print(f'Visualizer started with PID {demo_proc.pid}.')
print('Open the embedded noVNC pane above, then press F4 inside the viewer for the sensor panel.')


## Optional helpers

- The setup cell automatically selects the `cp311` or `cp312` wheel based on the active Colab runtime.
- To use a newer published wheel later, update `RELEASE_TAG` in the second cell.
- Re-run the **noVNC** cell if the browser frame disconnects.
- Change `--light` to `--heavy` in the launch cell to run the heavier payload scenario.
- If you want to stop the current viewer, run:

```python
import os, signal
os.kill(demo_proc.pid, signal.SIGTERM)
```
